# Soil–Vegetation Analysis

This notebook rebuilds the soil–vegetation analysis from the curated
`soil_aoi_ready.gpkg`.

The curated GeoPackage contains the validated soil layer together with
two contextual layers:

- `missing_water`: areas identified as water and therefore excluded
  from the soil coverage.
- `remaining_gaps`: residual areas not assigned to a soil polygon.

Previously identified code-00 areas caused by overlaps between soil
polygons are not treated as independent gaps in this analysis.

The vegetation dataset will subsequently be intersected with the
curated soil layer to determine the relationship between soil type
and vegetation type.

## 1. Project and Input Paths

This section imports the Python libraries required for working with
vector geospatial data, tabular data, and file paths.

We use GeoPandas for spatial data, Pandas for tabular data, and
Pathlib for reliable filesystem paths.

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# ------------------------------------------------------------------
# Input / processed data
# ------------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_TABLES_DIR = PROCESSED_DIR / "tables"
PROCESSED_VECTOR_DIR = PROCESSED_DIR / "vector"


# ------------------------------------------------------------------
# Output products
# ------------------------------------------------------------------

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

MAPS_DIR = OUTPUTS_DIR / "maps"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_OUTPUT_DIR = OUTPUTS_DIR / "table"
STATISTICS_DIR = OUTPUTS_DIR / "statistics"

# Create output directories if they do not already exist.
for directory in [
    PROCESSED_TABLES_DIR,
    PROCESSED_VECTOR_DIR,
    MAPS_DIR,
    FIGURES_DIR,
    TABLES_OUTPUT_DIR,
    STATISTICS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Input files
# ------------------------------------------------------------------

soil_ready_path = (
    PROCESSED_VECTOR_DIR / "soil_aoi_ready.gpkg"
)

vegetation_path = (
    RAW_DIR / "land_cover" / "landcover_vegetation.shp"
)

print("Project root:", PROJECT_ROOT)
print("Soil GeoPackage:", soil_ready_path)
print("Vegetation:", vegetation_path)

## 2. Inspect the Layers in the Curated Soil GeoPackage

Before loading the data, inspect the available GeoPackage layers.

This confirms that the curated file contains the expected soil,
water, and remaining-gap layers and prevents us from assuming a layer
name or structure that may not exist.

In [ ]:
layers = gpd.list_layers(soil_ready_path)

layers

## 3. Load the Curated Soil and Gap Layers

Load the three relevant layers from the curated GeoPackage.

`soils` is the primary analytical soil layer.

`missing_water` and `remaining_soil_gaps` are retained as separate
contextual layers. They are not treated as soil classes.

The previously investigated code-00 areas resulting from overlaps
between soil polygons are not included as an additional gap category.

In [ ]:
soil_aoi_ready = gpd.read_file(
    soil_ready_path,
    layer="soils"
)

missing_water = gpd.read_file(
    soil_ready_path,
    layer="missing_water"
)

remaining_soil_gaps = gpd.read_file(
    soil_ready_path,
    layer="remaining_soil_gaps"
)

### 3.1 Inspect the Basic Structure of the Loaded Layers

Check the number of features, coordinate reference system, geometry
type, and attribute fields for each layer.

This confirms that the layers were loaded correctly and establishes
the spatial reference and data structure before performing any spatial
operations.

In [ ]:
for name, gdf in {
    "soils": soil_aoi_ready,
    "missing_water": missing_water,
    "remaining_soil_gaps": remaining_soil_gaps,
}.items():

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Features:  {len(gdf)}")
    print(f"CRS:       {gdf.crs}")
    print(f"Geometry:  {gdf.geometry.geom_type.unique().tolist()}")
    print(f"Columns:   {gdf.columns.tolist()}")

## 4. Inspect the Soil Attributes

The curated soil layer is the authoritative source for soil
classification in this analysis.

Inspect its columns and a small sample of records before proceeding.

We do not reconstruct soil classes from the previous lookup tables;
the curated layer is the starting point for the new analysis.

In [ ]:
print(soil_aoi_ready.columns.tolist())

soil_aoi_ready.head()

### 4.1 Validate the Soil Geometry

Before performing any spatial intersection, verify that the curated
soil geometries are valid.

A valid geometry is important because invalid polygons can cause
errors or unexpected results during spatial operations such as
intersection and overlay.

We will report the number of valid and invalid geometries without
modifying the layer at this stage.

In [ ]:
soil_valid = soil_aoi_ready.geometry.is_valid

print(f"Valid geometries:   {soil_valid.sum()}")
print(f"Invalid geometries: {(~soil_valid).sum()}")

### 4.2 Check for Missing Soil Names

Inspect whether any soil polygons lack a `soil_name`.

A missing soil name is different from a spatial gap: the polygon
exists geometrically, but its soil classification is absent.

We should identify such features before using `soil_name` as the
primary soil classification variable.

In [ ]:
missing_soil_name = soil_aoi_ready[
    soil_aoi_ready["soil_name"].isna()
    | soil_aoi_ready["soil_name"].astype(str).str.strip().eq("")
]

print(f"Features without soil_name: {len(missing_soil_name)}")

if len(missing_soil_name) > 0:
    display(missing_soil_name)

## 5. Check the Soil Classification Inventory

List the distinct soil names and count how many polygons belong to
each soil class.

This provides the initial soil-class inventory that will later be
used to summarize the soil–vegetation relationships.

At this stage we are counting polygons, not calculating vegetation
areas.

In [ ]:
soil_class_counts = (
    soil_aoi_ready["soil_name"]
    .value_counts(dropna=False)
    .rename_axis("soil_name")
    .reset_index(name="polygon_count")
)

soil_class_counts

### 5.1 Save Soil Class Inventory

In [ ]:
soil_class_counts_path = (
    PROCESSED_TABLES_DIR / "soil_class_counts.csv"
)

soil_class_counts.to_csv(
    soil_class_counts_path,
    index=False,
)

print("Saved:", soil_class_counts_path)

## 6. Create the Classified Soil Subset

Create a separate view containing only polygons with an assigned
`soil_name`.

The undefined 0/0 polygon remains in `soil_aoi_ready`, but it is
excluded from the soil-type × vegetation analysis.

This distinction allows us to preserve the complete curated layer
while ensuring that the analytical soil classes represent only
classified soil polygons.

In [ ]:
soil_classified = soil_aoi_ready[
    soil_aoi_ready["soil_name"].notna()
].copy()

unclassified_soil = soil_aoi_ready[
    soil_aoi_ready["soil_name"].isna()
].copy()

print(f"Total soil features:       {len(soil_aoi_ready)}")
print(f"Classified soil features:  {len(soil_classified)}")
print(f"Undefined 0/0 features:    {len(soil_aoi_ready) - len(soil_classified)}")

print(
    f"\nClassified soil area: "
    f"{soil_classified['area_km2'].sum():.3f} km²"
)

print(
    f"Unclassified soil area: "
    f"{unclassified_soil.geometry.area.sum() / 1_000_000:.3f} km²"
)

## 7. Load the Vegetation Dataset

Load the vegetation/land-cover dataset used in the previous workflow.

Unlike the soil layer, this dataset has not been curated specifically
for the current soil–vegetation analysis. We therefore inspect its
structure and spatial reference before using it.

The vegetation dataset will subsequently be transformed into the same
coordinate reference system as the curated soil layer.

In [ ]:
vegetation = gpd.read_file(vegetation_path)

print(f"Features: {len(vegetation)}")
print(f"CRS:      {vegetation.crs}")
print(f"Geometry: {vegetation.geometry.geom_type.unique().tolist()}")

print("\nColumns:")
print(vegetation.columns.tolist())

### 7.1 Inspect the Vegetation Attributes

Inspect the vegetation attribute table to understand which fields
contain the land-cover and vegetation classifications.

We will use the actual fields present in the dataset rather than
assuming that every field from the previous workflow is required.

At this stage we are only examining the source data; no filtering or
classification is performed.

In [ ]:
vegetation.head()

## 8. Inspect the Vegetation Classification Fields

Identify the unique values and frequency of the principal vegetation
and land-cover classification fields.

The previous workflow used fields such as `LCNAME`, `NEWVEG`, and
`TNC_CLASS`. We inspect them here before deciding which field(s)
should represent vegetation in the new soil–vegetation analysis.

In [ ]:
for field in ["LCNAME", "NEWVEG", "TNC_CLASS"]:
    if field in vegetation.columns:
        print(f"\n{field}")
        print("-" * len(field))
        print(f"Unique values: {vegetation[field].nunique(dropna=False)}")
        display(
            vegetation[field]
            .value_counts(dropna=False)
            .rename_axis(field)
            .reset_index(name="feature_count")
        )

### 8.1 Check for Missing Vegetation Classifications

Determine whether the principal vegetation classification fields
contain missing values.

Missing classifications should be identified before the spatial
intersection because they can produce unclassified soil–vegetation
records later in the workflow.

In [ ]:
classification_fields = [
    field
    for field in ["LCNAME", "NEWVEG", "TNC_CLASS"]
    if field in vegetation.columns
]

missing_summary = pd.DataFrame({
    "field": classification_fields,
    "missing_count": [
        vegetation[field].isna().sum()
        for field in classification_fields
    ],
    "total_features": len(vegetation),
})

missing_summary["missing_percent"] = (
    missing_summary["missing_count"]
    / missing_summary["total_features"]
    * 100
)

missing_summary

### 8.2 Save Missing-Classification Summary

In [ ]:
missing_summary_path = (
    PROCESSED_TABLES_DIR / "vegetation_classification_missing_summary.csv"
)

missing_summary.to_csv(
    missing_summary_path,
    index=False,
)

print("Saved:", missing_summary_path)

## 9. Prepare the Vegetation Dataset for Spatial Analysis

The vegetation/land-cover dataset is an external reference dataset containing categorical vegetation and land-cover information.

The original vegetation source data are preserved unchanged. For spatial analysis, the vegetation geometries are repaired where necessary and transformed to the CRS used by the curated soil dataset (`soil_aoi_ready.gpkg`), EPSG:3795.

The prepared vegetation layer will subsequently be restricted to the Jaruco/Mayabeque study AOI and used as the spatial reference for assigning vegetation classes to the curated soil polygons.

This keeps the roles of the datasets clear:

- `soil_aoi_ready.gpkg` → curated soil analysis dataset and primary spatial framework.
- Vegetation dataset → external reference used to determine vegetation/land-cover associated with the soil polygons.
- Original vegetation source → preserved unchanged as source/reference data.


### 9.1 Make a Working Copy and Repair Geometry

Create a working copy of the vegetation dataset so that geometry repairs do not alter the original source data.


In [ ]:
from shapely.validation import make_valid

vegetation_analysis = vegetation.copy()

# Repair only invalid geometries
invalid_before = (~vegetation_analysis.geometry.is_valid).sum()

vegetation_analysis["geometry"] = (
    vegetation_analysis.geometry.apply(make_valid)
)

invalid_after = (~vegetation_analysis.geometry.is_valid).sum()

print("Invalid geometries before repair:", invalid_before)
print("Invalid geometries after repair:", invalid_after)

### 9.2 Inspect the Resulting Geometry Types

Check the geometry types after repair before continuing with reprojection and spatial operations.


In [ ]:
print(
    vegetation_analysis.geometry.geom_type.value_counts()
)

### 9.3 Reproject to the Soil Analysis CRS

Transform the working vegetation layer to the CRS used by the curated soil dataset.


In [ ]:
vegetation_analysis = vegetation_analysis.to_crs(
    soil_aoi_ready.crs
)

print("Vegetation CRS:", vegetation_analysis.crs)
print("Soil CRS:", soil_aoi_ready.crs)

### 9.4 Verify the Reprojection

Compare the transformed vegetation bounds with the curated soil bounds as a basic spatial sanity check.


In [ ]:
print("Vegetation bounds after reprojection:")
print(vegetation_analysis.total_bounds)

print("\nSoil bounds:")
print(soil_aoi_ready.total_bounds)

## 10. Create a Working Vegetation Layer Clipped to the Full AOI

Create a working vegetation layer for the study area using the geometry of the curated, classified soil layer.

The original vegetation source remains unchanged; only the working copy is restricted to the analysis AOI.


In [ ]:
# Create the full AOI geometry from the curated, classified soil layer.
# soil_classified contains only soil features with a valid soil_name.

aoi_geometry = soil_classified.geometry.union_all()

# Create a GeoDataFrame representing the full AOI.
aoi = gpd.GeoDataFrame(
    {"geometry": [aoi_geometry]},
    crs=soil_classified.crs
)

# Select vegetation polygons that intersect the full AOI.
vegetation_aoi_candidates = vegetation_analysis[
    vegetation_analysis.geometry.intersects(aoi_geometry)
].copy()

# Clip the candidate vegetation polygons to the full AOI.
vegetation_aoi = gpd.clip(
    vegetation_aoi_candidates,
    aoi
)

print("Vegetation features before AOI clip:", len(vegetation))
print("Vegetation features intersecting AOI:", len(vegetation_aoi_candidates))
print("Vegetation features after AOI clip:", len(vegetation_aoi))
print("Vegetation CRS:", vegetation_aoi.crs)
print("AOI CRS:", aoi.crs)

## 11. Inspect the Vegetation Layer Within the AOI

Verify the vegetation features retained within the study area and confirm their classification attributes.


### 11.1 Inspect Vegetation Classes Within the AOI

Inspect the principal vegetation and land-cover classification fields for the retained features.


In [ ]:
# Inspect the vegetation features retained within the full AOI.

vegetation_aoi[
    ["LCNAME", "NEWVEG", "TNC_CLASS", "SPECIES_CO", "DISTURBED_"]
]

### 11.2 Count the Vegetation Classifications Within the AOI

Count the values in the principal classification fields to document which classes are represented in the study area.


In [ ]:
print("LCNAME:")
print(vegetation_aoi["LCNAME"].value_counts(dropna=False))

print("\nNEWVEG:")
print(vegetation_aoi["NEWVEG"].value_counts(dropna=False))

print("\nTNC_CLASS:")
print(vegetation_aoi["TNC_CLASS"].value_counts(dropna=False))

### 11.3 Validate Vegetation Geometry

Confirm geometry types, spatial bounds, and geometry validity before using the layer in an intersection.


In [ ]:
print("Geometry types:")
print(vegetation_aoi.geometry.geom_type.value_counts())

print("\nVegetation AOI bounds:")
print(vegetation_aoi.total_bounds)

print("\nInvalid geometries:")
print((~vegetation_aoi.geometry.is_valid).sum())

## 12. Validate Vegetation Coverage Against the AOI

Before intersecting vegetation with the soil layer, verify whether the retained vegetation polygons collectively cover the analysis AOI and whether they contain overlaps or gaps.

This establishes whether the vegetation reference provides complete spatial coverage for the area used in the soil–vegetation analysis.


In [ ]:
# Calculate the AOI area and total vegetation area.

aoi_area = aoi_geometry.area
vegetation_area = vegetation_aoi.geometry.area.sum()

print(f"AOI area: {aoi_area / 1_000_000:.3f} km²")
print(f"Vegetation area: {vegetation_area / 1_000_000:.3f} km²")
print(
    f"Difference: "
    f"{(aoi_area - vegetation_area) / 1_000_000:.3f} km²"
)

In [ ]:
# Check for overlapping vegetation polygons within the AOI.

vegetation_union = vegetation_aoi.geometry.union_all()

vegetation_overlap_area = (
    vegetation_aoi.geometry.area.sum()
    - vegetation_union.area
)

print(
    f"Vegetation overlap area: "
    f"{vegetation_overlap_area / 1_000_000:.6f} km²"
)

In [ ]:
# Calculate any remaining gap between the AOI and the vegetation layer.

vegetation_gap = aoi_geometry.difference(vegetation_union)

print(
    f"Vegetation gap area: "
    f"{vegetation_gap.area / 1_000_000:.6f} km²"
)

print("Gap geometry type:", vegetation_gap.geom_type)

### 12.1 Save Vegetation Coverage Validation for Presentation

Save the key vegetation coverage checks as a compact CSV in `outputs/statistics` so the values can be reused in the project PDF or PowerPoint without manually transcribing notebook output.


In [ ]:
vegetation_coverage_summary = pd.DataFrame(
    {
        "metric": [
            "AOI area",
            "Vegetation area",
            "Area difference",
            "Vegetation overlap area",
            "Vegetation gap area",
        ],
        "value_km2": [
            aoi_area / 1_000_000,
            vegetation_area / 1_000_000,
            (aoi_area - vegetation_area) / 1_000_000,
            vegetation_overlap_area / 1_000_000,
            vegetation_gap.area / 1_000_000,
        ],
    }
)

vegetation_coverage_output = (
    STATISTICS_DIR / "vegetation_coverage_validation.csv"
)

vegetation_coverage_summary.to_csv(
    vegetation_coverage_output,
    index=False,
)

print("Saved:", vegetation_coverage_output)


## 13. Prepare the Vegetation Classes for Soil Intersection

Create a clean working vegetation layer containing the classification information that will be transferred to the soil polygons.

The three principal classification fields (`LCNAME`, `NEWVEG`, and `TNC_CLASS`) are retained so that the original classification information remains available for comparison. `SPECIES_CO` and `DISTURBED_` are also retained as reference attributes.


In [ ]:
# Create the working vegetation layer for the soil–vegetation analysis.
# Keep the original classification fields available for comparison.

vegetation_for_intersection = vegetation_aoi[
    [
        "LCNAME",
        "NEWVEG",
        "TNC_CLASS",
        "SPECIES_CO",
        "DISTURBED_",
        "geometry"
    ]
].copy()

print("Features:", len(vegetation_for_intersection))
print("CRS:", vegetation_for_intersection.crs)
print("Geometry types:")
print(vegetation_for_intersection.geometry.geom_type.value_counts())

In [ ]:
# Confirm that the working vegetation layer has no missing
# values in the primary vegetation classification fields.

print(
    vegetation_for_intersection[
        ["LCNAME", "NEWVEG", "TNC_CLASS"]
    ].isna().sum()
)

## 14. Intersect the Soil and Vegetation Layers

Perform the spatial intersection between the curated classified soil layer and the validated vegetation reference layer.

The result represents soil × vegetation combinations. A soil polygon crossing a vegetation boundary is divided into separate features, with each resulting feature retaining the relevant soil and vegetation attributes.


In [ ]:
# Intersect the curated soil polygons with the vegetation reference layer.

soil_vegetation = gpd.overlay(
    soil_classified,
    vegetation_for_intersection,
    how="intersection"
)

print("Soil features before intersection:", len(soil_classified))
print("Vegetation features:", len(vegetation_for_intersection))
print("Soil–vegetation features:", len(soil_vegetation))
print("CRS:", soil_vegetation.crs)
print("Geometry types:")
print(soil_vegetation.geometry.geom_type.value_counts())

In [ ]:
# Validate the resulting soil–vegetation geometries.

print(
    "Invalid geometries:",
    (~soil_vegetation.geometry.is_valid).sum()
)

print("\nColumns:")
print(soil_vegetation.columns.tolist())

In [ ]:
# Check missing values in the principal soil and vegetation classifications.

check_fields = [
    "soil_name",
    "LCNAME",
    "NEWVEG",
    "TNC_CLASS"
]

print(
    soil_vegetation[check_fields]
    .isna()
    .sum()
)

In [ ]:
# Inspect the distinct soil–vegetation combinations.

print(
    "Unique soil names:",
    soil_vegetation["soil_name"].nunique()
)

print(
    "Unique LCNAME classes:",
    soil_vegetation["LCNAME"].nunique()
)

print(
    "Unique NEWVEG classes:",
    soil_vegetation["NEWVEG"].nunique()
)

print(
    "Unique TNC_CLASS classes:",
    soil_vegetation["TNC_CLASS"].nunique()
)

## 15. Investigate the Dropped Intersection Geometries

The initial intersection reported geometry types different from the expected polygonal output. Rerun the operation with `keep_geom_type=False` so that these geometries can be inspected rather than ignored.


In [ ]:
# Re-run the intersection while retaining all geometry types.
# This is diagnostic only; it does not modify soil_classified
# or vegetation_for_intersection.

soil_vegetation_all = gpd.overlay(
    soil_classified,
    vegetation_for_intersection,
    how="intersection",
    keep_geom_type=False
)

print("Features with all geometry types:", len(soil_vegetation_all))

print("\nGeometry types:")
print(
    soil_vegetation_all.geometry.geom_type.value_counts()
)

In [ ]:
# Compare the feature counts from the two intersection approaches.

print("With keep_geom_type=True :", len(soil_vegetation))
print("With keep_geom_type=False:", len(soil_vegetation_all))
print(
    "Additional geometries:",
    len(soil_vegetation_all) - len(soil_vegetation)
)

In [ ]:
# Identify geometries that are not Polygon or MultiPolygon.

non_polygon = soil_vegetation_all[
    ~soil_vegetation_all.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

print("Non-polygon geometries:", len(non_polygon))

if len(non_polygon) > 0:
    print(
        non_polygon[
            ["soil_name", "LCNAME", "NEWVEG", "TNC_CLASS", "geometry"]
        ]
    )

## 16. Normalize the Geometry Collections

The diagnostic intersection identified GeometryCollection results containing polygonal and non-polygonal components.

For the soil–vegetation inventory, only polygonal components represent areas of interest. Extract those polygonal components while preserving the associated attributes.


In [ ]:
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

def extract_polygon_geometry(geom):
    """
    Return only polygonal components from a geometry.
    Polygon and MultiPolygon geometries are returned unchanged.
    GeometryCollections are reduced to their Polygon/MultiPolygon parts.
    """
    
    if geom.geom_type in ["Polygon", "MultiPolygon"]:
        return geom
    
    if geom.geom_type == "GeometryCollection":
        polygon_parts = [
            part
            for part in geom.geoms
            if part.geom_type in ["Polygon", "MultiPolygon"]
        ]
        
        if not polygon_parts:
            return None
        
        if len(polygon_parts) == 1:
            return polygon_parts[0]
        
        polygons = []
        
        for part in polygon_parts:
            if part.geom_type == "Polygon":
                polygons.append(part)
            elif part.geom_type == "MultiPolygon":
                polygons.extend(list(part.geoms))
        
        return MultiPolygon(polygons)
    
    return None


# Apply the extraction to the intersection layer.
soil_vegetation["geometry"] = (
    soil_vegetation.geometry.apply(extract_polygon_geometry)
)

print("Geometry types after extraction:")
print(
    soil_vegetation.geometry.geom_type.value_counts()
)

print("\nEmpty geometries:")
print(soil_vegetation.geometry.is_empty.sum())

print("\nMissing geometries:")
print(soil_vegetation.geometry.isna().sum())

print("\nInvalid geometries:")
print((~soil_vegetation.geometry.is_valid).sum())

In [ ]:
# Inspect the two features that were originally GeometryCollections.

geometry_collection_rows = [77, 102]

print(
    soil_vegetation.loc[
        geometry_collection_rows,
        [
            "soil_name",
            "LCNAME",
            "NEWVEG",
            "TNC_CLASS",
            "geometry"
        ]
    ]
)

In [ ]:
# Calculate the areas of the two normalized features.

for idx in geometry_collection_rows:
    area_km2 = (
        soil_vegetation.loc[idx, "geometry"].area
        / 1_000_000
    )
    
    print(
        f"Feature {idx}: {area_km2:.6f} km²"
    )

## 17. Validate Soil–Vegetation Area Coverage

Verify that the validated soil–vegetation layer represents the same classified soil area, with no unintended uncovered soil and no excess area outside the classified soil framework.


In [ ]:
# Calculate the total area of the soil and soil–vegetation layers.

soil_area = soil_classified.geometry.area.sum()
soil_vegetation_area = soil_vegetation.geometry.area.sum()

print(f"Soil classified area: {soil_area / 1_000_000:.6f} km²")
print(
    f"Soil–vegetation area: "
    f"{soil_vegetation_area / 1_000_000:.6f} km²"
)
print(
    f"Area difference: "
    f"{(soil_area - soil_vegetation_area) / 1_000_000:.6f} km²"
)

In [ ]:
# Union all soil–vegetation geometries.

soil_vegetation_union = (
    soil_vegetation.geometry.union_all()
)

# Calculate any area of the classified soil layer that is not
# represented in the soil–vegetation intersection.

soil_gap = (
    soil_classified.geometry.union_all()
    .difference(soil_vegetation_union)
)

print(
    f"Uncovered soil area: "
    f"{soil_gap.area / 1_000_000:.6f} km²"
)

In [ ]:
# Check for any soil–vegetation geometry extending outside
# the classified soil area.

soil_union = soil_classified.geometry.union_all()

soil_excess = (
    soil_vegetation_union.difference(soil_union)
)

print(
    f"Excess soil–vegetation area: "
    f"{soil_excess.area / 1_000_000:.6f} km²"
)

### 17.1 Save Soil–Vegetation Coverage Validation for Presentation

Save the final area-validation metrics as a compact CSV for direct use in the project PDF or PowerPoint.


In [ ]:
soil_vegetation_coverage_summary = pd.DataFrame(
    {
        "metric": [
            "Classified soil area",
            "Soil–vegetation area",
            "Area difference",
            "Uncovered soil area",
            "Excess soil–vegetation area",
        ],
        "value_km2": [
            soil_area / 1_000_000,
            soil_vegetation_area / 1_000_000,
            (soil_area - soil_vegetation_area) / 1_000_000,
            soil_gap.area / 1_000_000,
            soil_excess.area / 1_000_000,
        ],
    }
)

soil_vegetation_coverage_output = (
    STATISTICS_DIR / "soil_vegetation_coverage_validation.csv"
)

soil_vegetation_coverage_summary.to_csv(
    soil_vegetation_coverage_output,
    index=False,
)

print("Saved:", soil_vegetation_coverage_output)


## 18. Inspect the Soil–Vegetation Inventory

Inspect the validated intersection attributes before calculating statistics and exporting the final analytical layer.


In [ ]:
# Inspect the first records of the validated soil–vegetation layer.

soil_vegetation[
    [
        "soil_name",
        "area_km2",
        "LCNAME",
        "NEWVEG",
        "TNC_CLASS",
        "SPECIES_CO",
        "DISTURBED_"
    ]
].head(10)

## 19. Calculate Soil–Vegetation Feature Areas

Calculate the area of each validated soil–vegetation feature directly from its geometry in the projected CRS.


In [ ]:
# Calculate the area of each soil–vegetation polygon.

soil_vegetation["area_km2"] = (
    soil_vegetation.geometry.area / 1_000_000
)

print(
    soil_vegetation["area_km2"].describe()
)

In [ ]:
print(
    f"Soil–vegetation total area: "
    f"{soil_vegetation['area_km2'].sum():.6f} km²"
)

print(
    f"Classified soil total area: "
    f"{soil_classified.geometry.area.sum() / 1_000_000:.6f} km²"
)

## 20. Inspect the Soil × Vegetation Combinations

Summarize the number of features and total area for each soil × vegetation combination. This provides the main analytical table used to describe the relationship between soil classes and vegetation classes.


In [ ]:
# Summarize the number of features and area for each
# soil–vegetation combination.

soil_vegetation_summary = (
    soil_vegetation
    .groupby(["soil_name_en", "LCNAME"], dropna=False)
    .agg(
        feature_count=("geometry", "size"),
        area_km2=("area_km2", "sum")
    )
    .reset_index()
    .sort_values(
        ["soil_name_en", "area_km2"],
        ascending=[True, False]
    )
)

soil_vegetation_summary

### 20.1 Save the Analytical Summary for Presentation

Save a copy of the soil × vegetation summary in `data/processed/tables` so the final report and presentation can use the same validated analytical table as the notebook.


In [ ]:
soil_vegetation_summary_path = (
    PROCESSED_TABLES_DIR / "soil_vegetation_summary.csv"
)

soil_vegetation_summary.to_csv(
    soil_vegetation_summary_path,
    index=False,
)

print("Saved:", soil_vegetation_summary_path)

## 21. Export the Validated Soil–Vegetation Layer

Save the validated soil–vegetation analytical layer as a GeoPackage in the project's processed vector directory.


In [ ]:
# Export the validated soil–vegetation intersection to GeoPackage.

soil_vegetation_output = (
    PROCESSED_VECTOR_DIR / "soil_vegetation_analysis.gpkg"
)

soil_vegetation.to_file(
    soil_vegetation_output,
    layer="soil_vegetation",
    driver="GPKG"
)

print("Saved:", soil_vegetation_output)
print("Features:", len(soil_vegetation))
print("CRS:", soil_vegetation.crs)

In [ ]:
# Reload the exported layer and verify the saved result.

soil_vegetation_check = gpd.read_file(
    soil_vegetation_output,
    layer="soil_vegetation"
)

print("Features:", len(soil_vegetation_check))
print("CRS:", soil_vegetation_check.crs)

print("\nGeometry types:")
print(
    soil_vegetation_check.geometry.geom_type.value_counts()
)

print("\nInvalid geometries:")
print(
    (~soil_vegetation_check.geometry.is_valid).sum()
)

print("\nTotal area:")
print(
    f"{soil_vegetation_check.geometry.area.sum() / 1_000_000:.6f} km²"
)

print("\nColumns:")
print(soil_vegetation_check.columns.tolist())

### 21.1 Verify the Exported Analytical Layer

Reload the saved GeoPackage and confirm the feature count, CRS, geometry types, geometry validity, and total area of the exported analytical layer.


In [ ]:
import geopandas as gpd

soil_vegetation_output = (
    PROCESSED_VECTOR_DIR / "soil_vegetation_analysis.gpkg"
)

# List the layers contained in the GeoPackage.
layers = gpd.list_layers(soil_vegetation_output)

print(layers)

### 21.2 Create a Feature Index Map

The exported soil–vegetation layer is plotted with a sequential feature index shown inside each polygon. The index identifies individual records in the final analytical layer and is used for spatial inspection and quality control. It is not the original soil identifier.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Path to the output GeoPackage
soil_vegetation_output = (
    PROCESSED_VECTOR_DIR / "soil_vegetation_analysis.gpkg"
)

# Read the soil–vegetation layer
soil_vegetation_plot = gpd.read_file(
    soil_vegetation_output,
    layer="soil_vegetation"
).reset_index(drop=True)

# Create the plot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot polygon boundaries
soil_vegetation_plot.boundary.plot(
    ax=ax,
    linewidth=0.7
)

# Add the index inside each polygon
for idx, row in soil_vegetation_plot.iterrows():

    # Point guaranteed to be inside the polygon
    label_point = row.geometry.representative_point()

    ax.annotate(
        str(idx),
        xy=(label_point.x, label_point.y),
        ha="center",
        va="center",
        fontsize=7
    )

# Plot formatting
ax.set_title(
    "Soil–Vegetation Analysis Polygons — Feature Index"
)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()

feature_index_map_path = (
    MAPS_DIR / "soil_vegetation_feature_index.png"
)

fig.savefig(
    feature_index_map_path,
    dpi=300,
    bbox_inches="tight",
)


plt.show()

## 22. Create a Soil–Vegetation Feature Index Map

Create a reference map showing the feature index of the validated soil–vegetation layer. This map is useful for identifying individual analytical polygons in the report and presentation.


In [ ]:

import geopandas as gpd
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# 1. Read the curated soil layer
# ---------------------------------------------------------

soil_ready_path = (
    PROCESSED_VECTOR_DIR / "soil_aoi_ready.gpkg"
)

soil_aoi_ready = gpd.read_file(
    soil_ready_path,
    layer="soils"
).reset_index(drop=True)


# ---------------------------------------------------------
# 2. Read contextual layers
# ---------------------------------------------------------

missing_water = gpd.read_file(
    soil_ready_path,
    layer="missing_water"
)

remaining_soil_gaps = gpd.read_file(
    soil_ready_path,
    layer="remaining_soil_gaps"
)


# ---------------------------------------------------------
# 3. Define the soil colors
# ---------------------------------------------------------

soil_colors = {
    # Ferralítico Rojo
    "Ferralítico Rojo Típico": "#A63D2F",
    "Ferralítico Rojo Compactado": "#8F2F24",
    "Ferralítico Rojo Hidratado": "#B85A3A",
    "Ferralítico Rojo Antrópico": "#7F3028",

    # Ferralítico Amarillento
    "Ferralítico Amarillento Típico": "#C9A227",
    "Ferralítico Amarillento Concrecionario": "#A98217",

    # Fersialítico
    "Fersialítico Pardo Rojizo Típico": "#8B5A3C",

    # Pardo con Carbonatos
    "Pardo con Carbonatos Negro Grisáceo": "#5E5A50",
    "Pardo con Carbonatos Típico": "#806A4A",

    # Húmico Carbonático
    "Húmico Carbonático Gris amarillento": "#9A956F",
    "Húmico Carbonático Típico": "#756B4A",

    # Oscuro Plástico
    "Oscuro Plástico Gleyzado Negro Grisáceo": "#454545",
    "Oscuro Plástico no Gleyzado Gris Amarillento": "#77735C",

    # Rendzina
    "Rendzina Roja Típico": "#9B4935",

    # Gley Ferralítico
    "Gley Ferralítico Concrecionario": "#6B6045",

    # Aluvial
    "Aluvial Diferenciado": "#B58B5A",
    "Aluvial Poco Diferenciado": "#C7A77A",

    # Esquelético
    "Esquelético Diferenciado": "#C4B69A",
}


# ---------------------------------------------------------
# 4. Assign colors to soil polygons
# ---------------------------------------------------------
#
# NaN soil classifications are deliberately shown in a
# separate light-grey color.
#

plot_colors = (
    soil_aoi_ready["soil_name"]
    .map(soil_colors)
    .fillna("#BDBDBD")
)


# ---------------------------------------------------------
# 5. Check for unexpected soil classes
# ---------------------------------------------------------

missing_colors = (
    soil_aoi_ready.loc[
        soil_aoi_ready["soil_name"].notna()
        & soil_aoi_ready["soil_name"].map(soil_colors).isna(),
        "soil_name",
    ]
    .unique()
)

print("Soil classes without assigned colors:")

if len(missing_colors) == 0:
    print("None")
else:
    for soil in missing_colors:
        print(f"  - {soil}")


# ---------------------------------------------------------
# 6. Build the classified-soil mask
# ---------------------------------------------------------
#
# This is the critical part.
#
# Only soil polygons with an actual soil classification are
# allowed to receive vegetation boundary lines.
#
# The NaN soil polygon is therefore excluded from this mask.
#

classified_soil = soil_aoi_ready[
    soil_aoi_ready["soil_name"].notna()
]

classified_soil_union = classified_soil.geometry.union_all()


# ---------------------------------------------------------
# 7. Remove contextual gaps from the vegetation-boundary
#    drawing area
# ---------------------------------------------------------
#
# This guarantees that vegetation boundaries are not drawn
# across the remaining soil gaps or the missing-water area.
#

excluded_areas = (
    missing_water.geometry.union_all()
    .union(remaining_soil_gaps.geometry.union_all())
)

vegetation_boundary_area = (
    classified_soil_union.difference(excluded_areas)
)


# ---------------------------------------------------------
# 8. Keep only original vegetation boundaries that fall
#    inside valid classified soil
# ---------------------------------------------------------

vegetation_boundaries = (
    vegetation_aoi.boundary
    .intersection(vegetation_boundary_area)
)


# ---------------------------------------------------------
# 9. Create figure
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 10))


# ---------------------------------------------------------
# 10. Plot soil polygons
# ---------------------------------------------------------
#
# No soil polygon outlines.
#

soil_aoi_ready.plot(
    ax=ax,
    color=plot_colors,
    edgecolor="none",
)


# ---------------------------------------------------------
# 11. Plot missing water
# ---------------------------------------------------------

missing_water.plot(
    ax=ax,
    color="#1976D2",
    edgecolor="#0D47A1",
    linewidth=0.8,
    alpha=0.85,
)


# ---------------------------------------------------------
# 12. Plot remaining soil gaps
# ---------------------------------------------------------

remaining_soil_gaps.plot(
    ax=ax,
    color="#BDBDBD",
    edgecolor="#616161",
    linewidth=0.6,
    alpha=0.9,
)


# ---------------------------------------------------------
# 13. Plot vegetation boundaries
# ---------------------------------------------------------
#
# These are the original vegetation boundaries, but only
# where they overlap valid classified soil.
#

vegetation_boundaries.plot(
    ax=ax,
    color="#1B5E20",
    linewidth=1.5,
    alpha=1,
)


# ---------------------------------------------------------
# 14. Add feature index
# ---------------------------------------------------------
#
# Use the curated soil layer, so the index corresponds to
# soil_aoi_ready rather than the soil–vegetation intersection.
#

for idx, row in soil_aoi_ready.iterrows():

    label_point = row.geometry.representative_point()

    ax.annotate(
        str(idx),
        xy=(label_point.x, label_point.y),
        ha="center",
        va="center",
        fontsize=7,
    )


# ---------------------------------------------------------
# 15. Formatting
# ---------------------------------------------------------

ax.set_title(
    "Soil Types with Vegetation Boundaries — Feature Index"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()


# ---------------------------------------------------------
# 16. Save map
# ---------------------------------------------------------

soil_types_map_path = (
    MAPS_DIR / "soil_types_with_vegetation_boundaries.png"
)

fig.savefig(
    soil_types_map_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()



## 23. Create a Soil Type Map with Vegetation Boundaries

Create a presentation map showing soil classes as the main thematic layer, with vegetation boundaries emphasized only within valid classified soil areas.

Contextual areas are retained separately so that missing water, remaining soil gaps, and unclassified soil are not confused with classified soil. They are represented using the same neutral grey contextual color.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# ---------------------------------------------------------
# 1. Read the soil–vegetation layer
# ---------------------------------------------------------

soil_vegetation_output = (
    PROCESSED_VECTOR_DIR / "soil_vegetation_analysis.gpkg"
)

soil_vegetation_plot = gpd.read_file(
    soil_vegetation_output,
    layer="soil_vegetation"
).reset_index(drop=True)


# ---------------------------------------------------------
# 2. Read contextual layers
# ---------------------------------------------------------

soil_ready_path = (
    PROCESSED_VECTOR_DIR / "soil_aoi_ready.gpkg"
)

missing_water = gpd.read_file(
    soil_ready_path,
    layer="missing_water"
)

remaining_soil_gaps = gpd.read_file(
    soil_ready_path,
    layer="remaining_soil_gaps"
)


# ---------------------------------------------------------
# 3. Define vegetation colors
# ---------------------------------------------------------

vegetation_colors = {
    "Tall grassland": "#8DB255",
    "Mixed Planted/Cultivated woody and herbaceous crops": "#6F9E4A",
    "Secondary or succesional herbaceous and woody vegetation": "#3F7045",
    "Aquatic vegetation": "#4F9A8A",
}


# Assign a color to each vegetation feature
plot_colors = soil_vegetation_plot["NEWVEG"].map(
    vegetation_colors
)


# ---------------------------------------------------------
# 4. Define contextual layer color
# ---------------------------------------------------------
#
# Remaining soil gaps, unclassified soil, and missing water
# are represented using the same neutral grey.
# This color is also used for areas not covered by vegetation.
#

context_color = "#BDBDBD"


# ---------------------------------------------------------
# 5. Create figure
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 10))


# ---------------------------------------------------------
# 6. Plot remaining soil gaps
# ---------------------------------------------------------

remaining_soil_gaps.plot(
    ax=ax,
    color=context_color,
    edgecolor="none",
)


# ---------------------------------------------------------
# 7. Plot missing water
# ---------------------------------------------------------

missing_water.plot(
    ax=ax,
    color=context_color,
    edgecolor="none",
)


# ---------------------------------------------------------
# 8. Plot unclassified soil
# ---------------------------------------------------------
#
# Only soil polygons with soil_name = NaN are plotted here.
# They use the same grey as the other contextual areas.
#

soil_aoi_ready = gpd.read_file(
    soil_ready_path,
    layer="soils"
)

unclassified_soil = soil_aoi_ready[
    soil_aoi_ready["soil_name"].isna()
]

unclassified_soil.plot(
    ax=ax,
    color=context_color,
    edgecolor="none",
)


# ---------------------------------------------------------
# 9. Plot vegetation polygons
# ---------------------------------------------------------

soil_vegetation_plot.plot(
    ax=ax,
    color=plot_colors,
    edgecolor="black",
    linewidth=0.5,
)


# ---------------------------------------------------------
# 10. Add feature index
# ---------------------------------------------------------

for idx, row in soil_vegetation_plot.iterrows():

    label_point = row.geometry.representative_point()

    ax.annotate(
        str(idx),
        xy=(label_point.x, label_point.y),
        ha="center",
        va="center",
        fontsize=7,
    )


# ---------------------------------------------------------
# 11. Create vegetation and contextual legend
# ---------------------------------------------------------

legend_elements = [
    Patch(
        facecolor=color,
        edgecolor="black",
        label=vegetation,
    )
    for vegetation, color in vegetation_colors.items()
]

# Add contextual areas to the legend
legend_elements.append(
    Patch(
        facecolor=context_color,
        edgecolor="black",
        label="Not covered",
    )
)

ax.legend(
    handles=legend_elements,
    title="Vegetation Types and Coverage",
    loc="upper left",
bbox_to_anchor=(1.02, 1.0),
    frameon=True,
    fontsize=9,
    title_fontsize=10,
)


# ---------------------------------------------------------
# 12. Formatting
# ---------------------------------------------------------

ax.set_title(
    "Vegetation Types"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()


# ---------------------------------------------------------
# 13. Save map
# ---------------------------------------------------------

vegetation_map_path = (
    MAPS_DIR / "vegetation_types.png"
)

fig.savefig(
    vegetation_map_path,
    dpi=300,
    bbox_inches="tight",
)


# ---------------------------------------------------------
# 14. Display map
# ---------------------------------------------------------

plt.show()

print(f"Saved: {vegetation_map_path}")

# Workflow summary

This notebook establishes a reproducible soil–vegetation spatial analysis for the GeoSoil-Cuba Jaruco/Mayabeque AOI using the curated `soil_aoi_ready.gpkg` and an external vegetation/land-cover reference dataset.

### Project setup and curated soil framework

* Defined the project paths and inspected the layers available in the curated soil GeoPackage.
* Loaded the curated `soils`, `missing_water`, and `remaining_soil_gaps` layers.
* Verified the structure, coordinate reference system, geometry types, and attributes of the curated layers.
* Used the curated soil layer as the primary spatial framework for the analysis.
* Kept `missing_water` and `remaining_soil_gaps` as contextual layers rather than treating them as soil classes.
* Excluded polygons without an assigned `soil_name` from the soil-type × vegetation analysis while preserving them in the complete curated dataset.

### Soil classification inventory

* Inspected the curated soil attributes and verified the validity of the soil geometries.
* Identified soil polygons with missing `soil_name` values.
* Counted the distinct soil classes and their corresponding polygon features.
* Saved the soil-class inventory for reuse in the project.

### Vegetation source preparation

* Loaded the external vegetation/land-cover dataset without modifying the original source.
* Inspected its classification fields and identified the principal vegetation attributes used in the analysis.
* Checked the vegetation dataset for missing classification values.
* Created a working copy for spatial processing.
* Repaired invalid vegetation geometries where necessary.
* Reprojected the working vegetation layer to **NAD27 / Cuba Norte (EPSG:3795)**, the CRS used by the curated soil dataset.
* Verified the transformed spatial extent against the soil analysis framework.

### Vegetation preparation within the study area

* Restricted the working vegetation layer to the Jaruco/Mayabeque analysis area.
* Inspected the vegetation features retained within the AOI.
* Reviewed the vegetation and land-cover classifications represented in the study area.
* Verified geometry types, spatial bounds, and geometry validity before spatial intersection.
* Preserved the principal vegetation classification fields and additional reference attributes for the subsequent analysis.

### Vegetation coverage validation

* Compared the vegetation reference layer with the analysis AOI.
* Checked for differences between the vegetation coverage and the target analysis area.
* Investigated possible overlaps and uncovered areas before performing the soil–vegetation intersection.
* Saved the key coverage-validation metrics to `outputs/statistics` for reuse in the project PDF or PowerPoint presentation.

### Soil–vegetation spatial intersection

* Intersected the classified soil polygons with the prepared vegetation reference layer.
* Generated individual soil × vegetation features where vegetation boundaries divided soil polygons.
* Investigated non-polygon geometries produced during the overlay operation rather than silently discarding them.
* Extracted the polygonal components required for the area-based soil–vegetation analysis.
* Validated the resulting analytical layer against the classified soil area to confirm that the intersection did not introduce unintended spatial losses or excess coverage.

### Soil–vegetation analysis and validation

* Inspected the attributes of the validated soil–vegetation layer.
* Calculated the area of each analytical polygon using the projected EPSG:3795 coordinate system.
* Summarized the number of features and total area for each soil × vegetation combination.
* Saved the validated analytical summary in `data/processed/tables`.
* Exported the final polygonal soil–vegetation layer as `soil_vegetation_analysis.gpkg`.
* Reloaded the exported GeoPackage and verified its feature count, CRS, geometry types, geometry validity, and total area.

### Maps and presentation outputs

* Created a feature-index map to provide a spatial reference for individual soil–vegetation analytical polygons.
* Created a soil-type map with vegetation boundaries to show the spatial relationship between the two datasets.
* Represented contextual areas such as missing water, remaining soil gaps, and unclassified soil using a neutral grey rather than presenting them as soil classes.
* Saved high-resolution maps under `outputs/maps`.
* Saved selected validation statistics under `outputs/statistics` so that the project PDF and PowerPoint presentation can use the same values generated by the analysis.
* Preserved the processed analytical GeoPackage and summary tables as reproducible project outputs.



# Stage 2 Conclusions

The soil–vegetation analysis establishes a reproducible spatial relationship between the curated soil classification and the external vegetation reference within the study area.

Key conclusions supported by the workflow are:

- The curated soil dataset provides the primary spatial framework for the analysis.
- The vegetation dataset is treated as an external reference and is not modified at source; geometry repairs and reprojection are performed on a working copy.
- Vegetation coverage is explicitly checked for area differences, overlaps, and gaps before spatial intersection.
- The soil–vegetation intersection is independently validated against the classified soil area.
- Non-polygon intersection results are investigated and normalized rather than being silently ignored.
- The final analytical layer contains polygonal soil × vegetation units suitable for area-based statistical summaries and thematic mapping.
- Presentation-ready CSV statistics and high-resolution PNG maps are saved under `outputs`, allowing the PDF or PowerPoint presentation to use the same results generated by the analysis notebook.

These results should be interpreted as a **spatial overlay analysis**, not as evidence that vegetation classes cause or determine soil classes. The analysis identifies where the mapped soil and vegetation classifications coincide and quantifies their spatial extent.
